# V023: derivative-aware controller diagnostic

This independently derived training-only diagnostic adds a derivative response to the earlier proportional/integral feedback relation. It tests whether the missing response to changes in pressure helps explain control traces. Known pressure is used only to diagnose training breaths; this is not an input-only predictor or a pressure-MAE experiment.

Let e_t=q_t-p_t, s_t=d s_(t-1)+e_t, and u_t=Kp e_t+Ki s_t+Kd(e_t-e_(t-1)). Eliminating s gives coefficients d, A=Kp+Ki+Kd, B=d Kp+(1+d)Kd, and C=d Kd in u_t=d u_(t-1)-A p_t+B p_(t-1)-C p_(t-2)+reference terms. PI is the nested case Kd=0. For a linear reference q_t=q0+slope*t, the reference terms are D*q0+[d*Kp+(1-d)*Kd]*slope+D*slope*t with D=A-B+C.

All PI/PID comparisons use identical eligible rows and chronological cuts. Coefficients are selected using fitting controls only; later controls diagnose the chosen coefficients. Pressure values remain known throughout this training diagnostic, so no result is a claim about unseen-pressure recovery.

In [ ]:
from pathlib import Path
import numpy as np

V023_FAMILIES=('pi_constant','pi_linear','pid_constant','pid_linear')

def controller_design_v023(u,p,phase,lag,family):
    u,p,phase=np.asarray(u,dtype=np.float64),np.asarray(p,dtype=np.float64),np.asarray(phase)
    if any(a.shape!=(80,) for a in (u,p,phase)) or not all(np.isfinite(a).all() for a in (u,p,phase)):
        raise ValueError('Each controller diagnostic must contain one finite 80-step breath')
    if family not in V023_FAMILIES or lag not in (0,1) or not np.isin(phase,[0,1]).all():
        raise ValueError('Unknown frozen family/lag/phase')
    k=np.arange(2+lag,80);n=k-lag
    valid=(phase[k]==0)&(phase[k-1]==0)&(phase[n]==0)&(phase[n-1]==0)&(phase[n-2]==0)
    valid&=(u[k]>1e-6)&(u[k]<99.999999)&(u[k-1]>1e-6)&(u[k-1]<99.999999)
    k=k[valid];n=k-lag
    columns=[u[k-1],-p[n],p[n-1]]
    if family.startswith('pid'):columns.append(-p[n-2])
    columns.append(np.ones(len(k)))
    if family.endswith('linear'):columns.append(n.astype(np.float64))
    return np.column_stack(columns),u[k],k,n

def physical_coefficients_v023(coef,family,reference_steps,tol=1e-6):
    coef=np.asarray(coef,dtype=np.float64)
    expected=3+int(family.startswith('pid'))+1+int(family.endswith('linear'))
    if family not in V023_FAMILIES or coef.shape!=(expected,) or not np.isfinite(coef).all():
        raise ValueError('Coefficient schema changed')
    d,a,b=map(float,coef[:3])
    if family.startswith('pid'):
        c=float(coef[3]);intercept=float(coef[4]);timecoef=float(coef[5]) if family.endswith('linear') else 0.
    else:
        c=0.;intercept=float(coef[3]);timecoef=float(coef[4]) if family.endswith('linear') else 0.
    kd=c/d if abs(d)>1e-12 else np.nan
    kp=(b-(1+d)*kd)/d if abs(d)>1e-12 else np.nan
    ki=a-kp-kd;den=a-b+c
    slope=timecoef/den if abs(den)>1e-12 else np.nan
    q0=(intercept-(d*kp+(1-d)*kd)*slope)/den if abs(den)>1e-12 else np.nan
    q=q0+slope*np.asarray(reference_steps)
    physical=(1e-8<d<=1+tol and den>1e-8 and kp>1e-8 and ki>=-tol and kd>=-tol
              and np.isfinite(q).all() and len(q)>0 and q.min()>=-tol and q.max()<=100+tol)
    return bool(physical),dict(decay=d,kp=float(kp),ki=float(ki),kd=float(kd),q0=float(q0),slope=float(slope))

def fit_linear_v023(X,target,positions,maximum_condition=1e10):
    Xf=X[positions];yf=target[positions]
    scale=np.maximum(np.sqrt(np.mean(Xf**2,axis=0)),1e-12);Z=Xf/scale
    if np.linalg.matrix_rank(Z)<Z.shape[1]:return None
    condition=float(np.linalg.cond(Z))
    if not np.isfinite(condition) or condition>=maximum_condition:return None
    coef=np.linalg.lstsq(Z,yf,rcond=None)[0]/scale
    return coef,condition

def select_fitting_candidate_v023(X,target,cut,estimator,seed,draws=64,maximum_condition=1e10):
    if estimator not in ('ordinary_least_squares','training_only_RANSAC64'):
        raise ValueError('Unknown estimator')
    if not (X.shape[1]+2<=cut<len(target)):raise ValueError('Insufficient fitting or held controls')
    ordinary=fit_linear_v023(X,target,np.arange(cut),maximum_condition)
    if ordinary is None:return None
    candidates=[ordinary]
    if estimator=='training_only_RANSAC64':
        rng=np.random.default_rng(seed)
        for _ in range(draws):
            trial=fit_linear_v023(X,target,np.sort(rng.choice(cut,size=X.shape[1],replace=False)),maximum_condition)
            if trial is not None:candidates.append(trial)
    # Never inspect held-control target values or physical admissibility when ranking candidates.
    def key(trial):
        err=np.abs(X[:cut]@trial[0]-target[:cut])
        return int((err<1e-4).sum()),-float(np.median(err))
    return max(candidates,key=key)

def controller_selfcheck_v023():
    checks={}
    def check(name,value):
        if not bool(value):raise AssertionError(name)
        checks[name]=True
    rng=np.random.default_rng(20262301)
    for family in V023_FAMILIES:
        for lag in (0,1):
            kp,ki,kd,d,q0=.4,.08,(.12 if family.startswith('pid') else 0.),.9,25.
            slope=.02 if family.endswith('linear') else 0.
            p=15.+rng.normal(size=80);q=q0+slope*np.arange(80);e=q-p
            memory=0.;eprev=0.;base=[]
            for error in e:
                memory=d*memory+error
                base.append(kp*error+ki*memory+kd*(error-eprev));eprev=error
            u=np.asarray(base) if lag==0 else np.r_[5.,np.asarray(base)[:-1]]
            phase=np.zeros(80,dtype=np.int8)
            X,target,k,n=controller_design_v023(u,p,phase,lag,family)
            den=(1-d)*kp+ki;intercept=den*q0+(d*kp+(1-d)*kd)*slope
            expected=[d,kp+ki+kd,d*kp+(1+d)*kd]
            if family.startswith('pid'):expected.append(d*kd)
            expected.append(intercept)
            if family.endswith('linear'):expected.append(den*slope)
            expected=np.array(expected)
            tag=family+':lag'+str(lag)
            check(tag+':independent_PID_recurrence',len(k)>50 and np.max(np.abs(X@expected-target))<2e-12)
            physical,params=physical_coefficients_v023(expected,family,np.arange(80))
            check(tag+':physical_inverse',physical and all(abs(params[key]-v)<1e-10 for key,v in dict(kp=kp,ki=ki,kd=kd,decay=d,q0=q0,slope=slope).items()))
            cut=int(.7*len(target));selected=select_fitting_candidate_v023(X,target,cut,'ordinary_least_squares',7)
            check(tag+':fit_recovers_coefficients',selected is not None and np.max(np.abs(selected[0]-expected))<1e-8)
            selected=select_fitting_candidate_v023(X,target,cut,'training_only_RANSAC64',7)
            changed=target.copy();changed[cut:]+=100
            other=select_fitting_candidate_v023(X,changed,cut,'training_only_RANSAC64',7)
            check(tag+':selection_excludes_held_controls',np.array_equal(selected[0],other[0]))
    p=15.+rng.normal(size=80);u=20.+rng.normal(size=80);phase=np.zeros(80,np.int8)
    u[[8,12]]=[0.,100.];phase[30:]=1
    X,target,k,n=controller_design_v023(u,p,phase,0,'pid_constant')
    check('clipped_current_and_previous_controls_excluded',not np.isin(k,[8,9,12,13]).any())
    check('phase_and_lag_rows_within_one_breath',np.all(k<30) and np.all(n-2>=0))
    try:controller_design_v023(u[:40],p,phase,0,'pid_constant')
    except ValueError:checks['partial_breath_rejected']=True
    else:raise AssertionError('partial_breath_rejected')
    check('rank_deficient_design_rejected',fit_linear_v023(np.ones((12,5)),np.ones(12),np.arange(8)) is None)
    bad=np.array([1.2,.6,.5,.1,2.])
    check('unstable_decay_rejected',not physical_coefficients_v023(bad,'pid_constant',np.arange(80))[0])
    bad=np.array([.9,.1,.5,.108,2.])
    check('negative_integral_gain_rejected',not physical_coefficients_v023(bad,'pid_constant',np.arange(80))[0])
    return checks
